# Experiment 7b — OOF blend study

Inputs: the two OOF CSVs produced by experiment 7a.

- `oof_predictions_7a_tabular.csv` — `pred_lgbm`, `pred_mlp` per (series_id, timestamp, fold)
- `oof_predictions_7a_seq2seq.csv` — `pred_lgbm`, `pred_seq` per (series_id, timestamp, fold), plus `lead_h`

Question this notebook answers (decision-shaped, thresholds set in §0 before any numbers are computed):

1. Does an **optimal-weight blend of LGBM + MLP** beat LGBM solo by more than the 0.05 WAPE noise floor? *(Almost certainly yes — the 50/50 blend in exp-7a-tabular already cleared this.)*
2. Does adding **seq2seq into a 3-way blend** improve on the 2-way blend by more than 0.05 WAPE? *(Open question — seq2seq is the weakest solo model but the most decorrelated.)*
3. Does **horizon-conditional blending** (different weights at different lead times) add anything? *(Diagnostic — seq2seq beat LGBM at lead-1h but lost elsewhere, so this is the regime where it could earn its keep.)*

The chosen blend's per-row OOF predictions are saved to `oof_predictions_7b_blend.csv` so that whichever notebook generates the leaderboard submission can re-use the fitted weights without re-running the blend search.

---
## §0. Configuration

In [11]:
from pathlib import Path

# Auto-locate the OOF CSVs whether the notebook is run from the experiment-7 folder,
# from the project root, or on Kaggle. First hit wins.
_CANDIDATES = [
    Path("."),
    Path("experiment-7"),
    Path("/kaggle/input/experiment-7-oof"),
    Path("/kaggle/input/datasets/xitij27/experiment-7"),
]
DATA_DIR = next(
    (p for p in _CANDIDATES if (p / "oof_predictions_7a_tabular.csv").exists()
                            and (p / "oof_predictions_7a_seq2seq.csv").exists()),
    None,
)
assert DATA_DIR is not None, (
    "Could not find oof_predictions_7a_tabular.csv + oof_predictions_7a_seq2seq.csv "
    f"in any of {[str(p) for p in _CANDIDATES]}"
)
OUTPUT_DIR = Path("/kaggle/working/")

OOF_TAB = DATA_DIR / "oof_predictions_7a_tabular.csv"
OOF_SEQ = DATA_DIR / "oof_predictions_7a_seq2seq.csv"

# ── Pre-registered decision thresholds ──────────────────────────────────────
# Same 0.05 WAPE noise floor used throughout exp-5 / exp-6 for "is this real".
GAIN_OVER_LGBM_FLOOR  = 0.05   # min CV-WAPE improvement vs LGBM solo to call a blend useful
GAIN_3WAY_OVER_2WAY   = 0.05   # min CV-WAPE improvement of 3-way over best 2-way to include seq2seq
WEIGHT_INCLUSION_FLOOR = 0.02  # if a model's optimal weight is below this, treat as effectively excluded

# Grid resolution for the weight search. 0.005 → 201 evals for 2-way, ~20k for 3-way.
GRID_STEP = 0.005

print(f"DATA_DIR    = {DATA_DIR.resolve()}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR.resolve()}")
print(f"Thresholds  : blend-vs-LGBM ≥ {GAIN_OVER_LGBM_FLOOR}, 3way-vs-2way ≥ {GAIN_3WAY_OVER_2WAY}")
print(f"Grid step   : {GRID_STEP}")

DATA_DIR    = /kaggle/input/datasets/xitij27/experiment-7
OUTPUT_DIR  = /kaggle/working
Thresholds  : blend-vs-LGBM ≥ 0.05, 3way-vs-2way ≥ 0.05
Grid step   : 0.005


---
## §1. Load OOF predictions and cross-validate the two files

Both notebooks reproduce the same exp-5.1 LightGBM-MAE baseline on the same 3-fold CV. So `pred_lgbm` and `target` should be identical between the two files for every row. If they aren't, the blend math is unsafe — fail loudly here rather than silently average inconsistent predictions later.

In [3]:
import numpy as np
import pandas as pd

df_tab = pd.read_csv(OOF_TAB, parse_dates=["timestamp"])
df_seq = pd.read_csv(OOF_SEQ, parse_dates=["timestamp"])

print(f"tabular file: {len(df_tab):,} rows, cols = {list(df_tab.columns)}")
print(f"seq2seq file: {len(df_seq):,} rows, cols = {list(df_seq.columns)}")

# Merge on (series_id, timestamp, fold). Keep both target / pred_lgbm copies for the cross-check.
df = df_tab.merge(
    df_seq[["series_id", "timestamp", "fold", "target", "pred_lgbm", "pred_seq", "lead_h"]]
        .rename(columns={"target": "target_seq", "pred_lgbm": "pred_lgbm_seq"}),
    on=["series_id", "timestamp", "fold"], how="inner",
)

assert len(df) == len(df_tab) == len(df_seq), (
    f"Row count diverged after merge: tab={len(df_tab)}, seq={len(df_seq)}, merged={len(df)}. "
    "The two notebooks did not produce predictions on the same OOF rows."
)

tgt_diff  = (df["target"] - df["target_seq"]).abs().max()
lgbm_diff = (df["pred_lgbm"] - df["pred_lgbm_seq"]).abs().max()
print(f"max |target_tab - target_seq|     = {tgt_diff:.3e}")
print(f"max |pred_lgbm_tab - pred_lgbm_seq| = {lgbm_diff:.3e}")

assert tgt_diff  < 1e-6, "target column disagrees between the two OOF files"
assert lgbm_diff < 1e-6, "pred_lgbm disagrees between the two OOF files — LGBM baselines didn't reproduce identically"

df = df.drop(columns=["target_seq", "pred_lgbm_seq"])
df = df[["series_id", "timestamp", "fold", "lead_h", "target", "pred_lgbm", "pred_mlp", "pred_seq"]]
print(f"\nUnified frame: {len(df):,} rows, {df.fold.nunique()} folds, {df.series_id.nunique()} series")
print(f"lead_h range: {df.lead_h.min()}..{df.lead_h.max()}")

tabular file: 96,768 rows, cols = ['series_id', 'timestamp', 'target', 'pred_lgbm', 'pred_mlp', 'fold']
seq2seq file: 96,768 rows, cols = ['series_id', 'timestamp', 'target', 'pred_lgbm', 'pred_seq', 'lead_h', 'fold']
max |target_tab - target_seq|     = 0.000e+00
max |pred_lgbm_tab - pred_lgbm_seq| = 0.000e+00

Unified frame: 96,768 rows, 3 folds, 96 series
lead_h range: 1..336


---
## §2. Solo CV-WAPE — reproduce the anchor values

Should match the headline numbers from the two 7a notebooks:

- LightGBM-MAE: **14.78 ± 0.62** (per-fold 14.08 / 15.58 / 14.68)
- MLP-MAE: **14.52 ± 0.59** (per-fold 13.88 / 15.30 / 14.39)
- Seq2seq LSTM: **16.30 ± 0.66** (per-fold 15.50 / 17.12 / 16.28)

If anything drifts here, the CSVs aren't what we think they are — stop and re-investigate.

In [4]:
def wape(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    return 100.0 * np.abs(y - yhat).sum() / np.abs(y).sum()

FOLDS = sorted(df.fold.unique().tolist())

def per_fold_wape(pred_col):
    return np.array([wape(df.loc[df.fold == k, "target"], df.loc[df.fold == k, pred_col]) for k in FOLDS])

solo = {col: per_fold_wape(col) for col in ["pred_lgbm", "pred_mlp", "pred_seq"]}

print(f"{'fold':>6}  {'LGBM':>7}  {'MLP':>7}  {'seq2seq':>9}")
for i, k in enumerate(FOLDS):
    print(f"  {k:>4}    {solo['pred_lgbm'][i]:>6.3f}  {solo['pred_mlp'][i]:>6.3f}  {solo['pred_seq'][i]:>8.3f}")
print("  ────")
print(f"  mean    {solo['pred_lgbm'].mean():>6.3f}  {solo['pred_mlp'].mean():>6.3f}  {solo['pred_seq'].mean():>8.3f}")
print(f"  std     {solo['pred_lgbm'].std():>6.3f}  {solo['pred_mlp'].std():>6.3f}  {solo['pred_seq'].std():>8.3f}")

LGBM_WAPE = solo["pred_lgbm"].mean()
MLP_WAPE  = solo["pred_mlp"].mean()
SEQ_WAPE  = solo["pred_seq"].mean()

# Sanity checks against the published anchors.
for name, val, expected in [("LGBM", LGBM_WAPE, 14.78), ("MLP", MLP_WAPE, 14.52), ("seq2seq", SEQ_WAPE, 16.30)]:
    if abs(val - expected) > 0.1:
        print(f"\n⚠️  {name} CV-WAPE = {val:.3f} drifted from anchor {expected:.2f} by {abs(val-expected):.2f}. Investigate before trusting the blend.")

  fold     LGBM      MLP    seq2seq
     0    14.082  13.875    15.503
     1    15.581  15.299    17.117
     2    14.683  14.394    16.276
  ────
  mean    14.782  14.523    16.299
  std      0.616   0.588     0.659


---
## §3. Naive equal-weight blends

Baselines to anchor the optimal-weight search. Three combinations:

- 50/50 LGBM + MLP
- 50/50 LGBM + seq2seq
- 1/3 each LGBM + MLP + seq2seq

Optimal weights in §4 should beat these or there's a bug in the optimizer.

In [5]:
def cv_wape_of_blend(weights, pred_cols):
    "Per-fold mean WAPE of `sum(w_i * pred_i)`. weights and pred_cols must align."
    blend = sum(w * df[c] for w, c in zip(weights, pred_cols))
    per_fold = [wape(df.loc[df.fold == k, "target"], blend[df.fold == k]) for k in FOLDS]
    return float(np.mean(per_fold)), np.array(per_fold)

naive_results = [
    ("LGBM solo                ", (1.0,),       ("pred_lgbm",)),
    ("MLP  solo                ", (1.0,),       ("pred_mlp",)),
    ("seq2seq solo             ", (1.0,),       ("pred_seq",)),
    ("50/50  LGBM + MLP        ", (0.5, 0.5),   ("pred_lgbm", "pred_mlp")),
    ("50/50  LGBM + seq2seq    ", (0.5, 0.5),   ("pred_lgbm", "pred_seq")),
    ("1/3·3  LGBM + MLP + seq  ", (1/3, 1/3, 1/3), ("pred_lgbm", "pred_mlp", "pred_seq")),
]

print(f"{'blend':<30}  {'CV-WAPE':>8}  {'Δ_vs_LGBM':>10}")
print("-" * 56)
for name, w, cols in naive_results:
    m, _ = cv_wape_of_blend(w, cols)
    print(f"  {name}  {m:>7.4f}  {LGBM_WAPE - m:>+9.4f}")

blend                            CV-WAPE   Δ_vs_LGBM
--------------------------------------------------------
  LGBM solo                  14.7819    +0.0000
  MLP  solo                  14.5226    +0.2593
  seq2seq solo               16.2988    -1.5169
  50/50  LGBM + MLP          14.4974    +0.2845
  50/50  LGBM + seq2seq      15.0389    -0.2570
  1/3·3  LGBM + MLP + seq    14.6764    +0.1055


---
## §4. Optimal pooled-weight blends

Brute-force grid search over the simplex `w_i ≥ 0, Σw_i = 1`, minimising **per-fold mean WAPE** (the metric we actually ship). Grid search instead of scipy because WAPE is L1-based (non-smooth) and the search space is 2D / 3D — at `GRID_STEP = 0.005` this is ~20k evaluations for the 3-way, fast enough on a laptop, and removes any concern about local minima or gradient noise.

Two blends are fit:

- **2-way**: LGBM + MLP — the floor case, expected to clear the 0.05 WAPE bar over LGBM solo.
- **3-way**: LGBM + MLP + seq2seq — the question is whether the seq2seq weight survives the optimizer and whether the 3-way beats the 2-way by more than `GAIN_3WAY_OVER_2WAY` (= 0.05 WAPE).

In [6]:
def grid_search_2way(col_a, col_b, step=GRID_STEP):
    ws = np.arange(0.0, 1.0 + step / 2, step)
    best_w, best_m, best_per_fold = None, float("inf"), None
    for w in ws:
        m, per_fold = cv_wape_of_blend((w, 1 - w), (col_a, col_b))
        if m < best_m:
            best_w, best_m, best_per_fold = w, m, per_fold
    return best_w, best_m, best_per_fold

def grid_search_3way(col_a, col_b, col_c, step=GRID_STEP):
    ws = np.arange(0.0, 1.0 + step / 2, step)
    best = (None, float("inf"), None)
    for w_a in ws:
        for w_b in np.arange(0.0, 1.0 - w_a + step / 2, step):
            w_c = 1.0 - w_a - w_b
            if w_c < -1e-9:
                continue
            w_c = max(w_c, 0.0)
            m, per_fold = cv_wape_of_blend((w_a, w_b, w_c), (col_a, col_b, col_c))
            if m < best[1]:
                best = ((w_a, w_b, w_c), m, per_fold)
    return best

import time

t0 = time.time()
w2_lgbm, blend2_mean, blend2_per_fold = grid_search_2way("pred_lgbm", "pred_mlp")
t2 = time.time() - t0
print(f"=== Optimal 2-way blend (LGBM + MLP) ===  [grid search {t2:.1f}s]")
print(f"  weights : LGBM = {w2_lgbm:.3f},  MLP = {1 - w2_lgbm:.3f}")
print(f"  per-fold WAPE: {[f'{w:.3f}' for w in blend2_per_fold]}")
print(f"  CV-WAPE : {blend2_mean:.4f}  (Δ vs LGBM solo = {LGBM_WAPE - blend2_mean:+.4f})")

t0 = time.time()
w3, blend3_mean, blend3_per_fold = grid_search_3way("pred_lgbm", "pred_mlp", "pred_seq")
t3 = time.time() - t0
print(f"\n=== Optimal 3-way blend (LGBM + MLP + seq2seq) ===  [grid search {t3:.1f}s]")
print(f"  weights : LGBM = {w3[0]:.3f},  MLP = {w3[1]:.3f},  seq2seq = {w3[2]:.3f}")
print(f"  per-fold WAPE: {[f'{w:.3f}' for w in blend3_per_fold]}")
print(f"  CV-WAPE : {blend3_mean:.4f}  (Δ vs LGBM solo = {LGBM_WAPE - blend3_mean:+.4f},  Δ vs 2-way = {blend2_mean - blend3_mean:+.4f})")

# Verdict on whether seq2seq earned its place.
print()
if w3[2] < WEIGHT_INCLUSION_FLOOR:
    print(f"✗  seq2seq weight = {w3[2]:.3f} < {WEIGHT_INCLUSION_FLOOR} — optimizer effectively excluded it. "
          f"3-way collapses to the 2-way.")
elif blend2_mean - blend3_mean < GAIN_3WAY_OVER_2WAY:
    print(f"~  seq2seq holds weight ({w3[2]:.3f}) but the 3-way gain ({blend2_mean - blend3_mean:+.4f}) "
          f"is below the {GAIN_3WAY_OVER_2WAY} threshold. Keep seq2seq as a diagnostic, ship the 2-way.")
else:
    print(f"✅ 3-way improves on the 2-way by {blend2_mean - blend3_mean:+.4f} WAPE > {GAIN_3WAY_OVER_2WAY}. "
          f"Include seq2seq in the final blend.")

=== Optimal 2-way blend (LGBM + MLP) ===  [grid search 0.9s]
  weights : LGBM = 0.285,  MLP = 0.715
  per-fold WAPE: ['13.828', '15.223', '14.354']
  CV-WAPE : 14.4683  (Δ vs LGBM solo = +0.3136)

=== Optimal 3-way blend (LGBM + MLP + seq2seq) ===  [grid search 97.1s]
  weights : LGBM = 0.265,  MLP = 0.685,  seq2seq = 0.050
  per-fold WAPE: ['13.828', '15.207', '14.353']
  CV-WAPE : 14.4624  (Δ vs LGBM solo = +0.3196,  Δ vs 2-way = +0.0059)

~  seq2seq holds weight (0.050) but the 3-way gain (+0.0059) is below the 0.05 threshold. Keep seq2seq as a diagnostic, ship the 2-way.


---
## §5. Per-fold-tuned weights — ablation

Refit the blend weights separately within each fold's OOF. This is **technically overfitting** to the OOF (the per-fold weights are chosen *using* the very rows the WAPE is computed on), so the resulting number is a *lower bound* on what a well-tuned per-fold blend could achieve, not an honest estimate. Useful only to answer: *if* fold-conditional weighting helps a lot, that suggests fitting weights per fold (or per regime) is worth a full inner-CV; if it barely helps, pooled weights are fine.

Threshold: if per-fold tuning beats pooled by less than 0.05 WAPE, pooled is the right choice.

In [7]:
def grid_search_3way_fold(fold_df, step=GRID_STEP):
    y = fold_df["target"].to_numpy()
    p_a = fold_df["pred_lgbm"].to_numpy()
    p_b = fold_df["pred_mlp"].to_numpy()
    p_c = fold_df["pred_seq"].to_numpy()
    ws = np.arange(0.0, 1.0 + step / 2, step)
    best = (None, float("inf"))
    for w_a in ws:
        for w_b in np.arange(0.0, 1.0 - w_a + step / 2, step):
            w_c = max(0.0, 1.0 - w_a - w_b)
            blend = w_a * p_a + w_b * p_b + w_c * p_c
            m = wape(y, blend)
            if m < best[1]:
                best = ((w_a, w_b, w_c), m)
    return best

print(f"{'fold':>4}  {'w_LGBM':>7}  {'w_MLP':>7}  {'w_seq':>7}  {'fold-tuned WAPE':>16}  {'pooled-w WAPE':>14}")
per_fold_tuned = []
for i, k in enumerate(FOLDS):
    fd = df[df.fold == k]
    (wa, wb, wc), m_tuned = grid_search_3way_fold(fd)
    per_fold_tuned.append(m_tuned)
    print(f"  {k:>2}    {wa:>6.3f}   {wb:>6.3f}   {wc:>6.3f}   {m_tuned:>14.4f}    {blend3_per_fold[i]:>12.4f}")

tuned_mean = float(np.mean(per_fold_tuned))
print(f"\n  per-fold-tuned CV-WAPE (LOWER BOUND, optimistic) : {tuned_mean:.4f}")
print(f"  pooled-weight 3-way CV-WAPE                       : {blend3_mean:.4f}")
print(f"  gap (tuned − pooled)                              : {blend3_mean - tuned_mean:+.4f}")

if blend3_mean - tuned_mean < 0.05:
    print("\n✅ Per-fold tuning barely helps. Pooled weights are the right choice — simpler, less overfit.")
else:
    print("\n~  Per-fold tuning shows a non-trivial gap. Could be real heterogeneity across folds, or just overfit")
    print("   to OOF noise. Resolving this honestly would need an inner CV (fit weights on one OOF, evaluate on a")
    print("   held-out OOF), which we haven't built. Park unless the gap > 0.1.")

fold   w_LGBM    w_MLP    w_seq   fold-tuned WAPE   pooled-w WAPE
   0     0.275    0.700    0.025          13.8264         13.8275
   1     0.275    0.640    0.085          15.2036         15.2070
   2     0.250    0.720    0.030          14.3511         14.3525

  per-fold-tuned CV-WAPE (LOWER BOUND, optimistic) : 14.4604
  pooled-weight 3-way CV-WAPE                       : 14.4624
  gap (tuned − pooled)                              : +0.0020

✅ Per-fold tuning barely helps. Pooled weights are the right choice — simpler, less overfit.


---
## §6. Per-horizon weight study

The seq2seq per-horizon table from exp-7a-seq2seq §8 showed seq2seq beating LGBM by 3.4 WAPE at lead-1h and losing by 1.3–1.7 WAPE at every other bucket. If that pattern holds in the blend, then horizon-conditional weights — small `w_seq` everywhere except lead-1, large `w_seq` at lead-1 — should outperform a single global weight set.

This is the same overfitting caveat as §5: per-bucket weights are fit on the same OOF rows we evaluate on. Treat the result as *upper bound on what horizon-conditional blending could deliver*.

In [8]:
BUCKETS = [(1, 1), (2, 6), (7, 24), (25, 72), (73, 168), (169, 336)]

def bucket_label(lo, hi):
    return f"{lo:>3}..{hi:<3}h"

print(f"{'lead':>10}  {'n':>6}  {'LGBM':>7}  {'MLP':>7}  {'seq':>7}  "
      f"{'w_LGBM':>7}  {'w_MLP':>7}  {'w_seq':>7}  {'blend':>7}")
print("-" * 88)

per_bucket_blend_wape = []
per_bucket_weights = {}
for lo, hi in BUCKETS:
    bd = df[(df.lead_h >= lo) & (df.lead_h <= hi)]
    if len(bd) == 0:
        continue
    y = bd["target"].to_numpy()
    p_lgbm = bd["pred_lgbm"].to_numpy()
    p_mlp  = bd["pred_mlp"].to_numpy()
    p_seq  = bd["pred_seq"].to_numpy()
    w_solo = (wape(y, p_lgbm), wape(y, p_mlp), wape(y, p_seq))
    # Grid search per bucket
    ws = np.arange(0.0, 1.0 + GRID_STEP / 2, GRID_STEP)
    best = (None, float("inf"))
    for w_a in ws:
        for w_b in np.arange(0.0, 1.0 - w_a + GRID_STEP / 2, GRID_STEP):
            w_c = max(0.0, 1.0 - w_a - w_b)
            m = wape(y, w_a * p_lgbm + w_b * p_mlp + w_c * p_seq)
            if m < best[1]:
                best = ((w_a, w_b, w_c), m)
    (wa, wb, wc), m_blend = best
    per_bucket_blend_wape.append((len(bd), m_blend))
    per_bucket_weights[(lo, hi)] = (wa, wb, wc)
    print(f"  {bucket_label(lo, hi):>10}  {len(bd):>6d}  "
          f"{w_solo[0]:>6.3f}  {w_solo[1]:>6.3f}  {w_solo[2]:>6.3f}  "
          f"{wa:>6.3f}  {wb:>6.3f}  {wc:>6.3f}  {m_blend:>6.3f}")

# Weighted average of per-bucket blend WAPE — apples-to-apples with the global blend.
n_total = sum(n for n, _ in per_bucket_blend_wape)
horizon_blend_pooled = sum(n * m for n, m in per_bucket_blend_wape) / n_total
print(f"\n  per-horizon-tuned WAPE (count-weighted) : {horizon_blend_pooled:.4f}")
print(f"  global 3-way pooled-weight WAPE         : {blend3_mean:.4f}")
print(f"  ↳ gap                                   : {blend3_mean - horizon_blend_pooled:+.4f}")
print("\n  Reminder: per-bucket weights are fit on the rows they're evaluated on. Treat the")
print("  gap as an *upper bound* on what horizon-conditional blending could yield with proper CV.")

      lead       n     LGBM      MLP      seq   w_LGBM    w_MLP    w_seq    blend
----------------------------------------------------------------------------------------
     1..1  h     288  24.052  24.684  20.673   0.140   0.000   0.860  20.001
     2..6  h    1440  21.573  21.093  22.857   0.050   0.730   0.220  20.751
     7..24 h    5184  12.514  12.252  14.244   0.220   0.685   0.095  12.183
    25..72 h   13824  17.178  16.759  18.427   0.175   0.735   0.090  16.706
    73..168h   27648  14.053  13.800  15.465   0.255   0.695   0.050  13.748
   169..336h   48384  14.670  14.452  16.295   0.310   0.675   0.015  14.387

  per-horizon-tuned WAPE (count-weighted) : 14.5287
  global 3-way pooled-weight WAPE         : 14.4624
  ↳ gap                                   : -0.0664

  Reminder: per-bucket weights are fit on the rows they're evaluated on. Treat the
  gap as an *upper bound* on what horizon-conditional blending could yield with proper CV.


---
## §7. Decision and saved artifact

Pick the blend that satisfies the pre-registered thresholds:

- LGBM solo is the *baseline* — anything we ship must beat it by ≥ 0.05 WAPE.
- Default ship is the 2-way LGBM + MLP optimal blend.
- Promote to 3-way (include seq2seq) only if the 3-way beats the 2-way by ≥ 0.05 WAPE *and* the seq2seq weight is ≥ 0.02.

The chosen blend's OOF predictions are written to `oof_predictions_7b_blend.csv` along with the weights used, so the submission notebook can apply the same weights to fresh test-time predictions.

In [12]:
include_seq = (w3[2] >= WEIGHT_INCLUSION_FLOOR) and (blend2_mean - blend3_mean >= GAIN_3WAY_OVER_2WAY)

if include_seq:
    chosen_weights = {"pred_lgbm": w3[0], "pred_mlp": w3[1], "pred_seq": w3[2]}
    chosen_wape = blend3_mean
    chosen_label = "3-way (LGBM + MLP + seq2seq)"
else:
    chosen_weights = {"pred_lgbm": w2_lgbm, "pred_mlp": 1.0 - w2_lgbm, "pred_seq": 0.0}
    chosen_wape = blend2_mean
    chosen_label = "2-way (LGBM + MLP)"

delta_vs_lgbm = LGBM_WAPE - chosen_wape
ships = delta_vs_lgbm >= GAIN_OVER_LGBM_FLOOR

print("=" * 60)
print("DECISION")
print("=" * 60)
print(f"Chosen blend  : {chosen_label}")
print(f"Weights       : LGBM = {chosen_weights['pred_lgbm']:.3f}, "
      f"MLP = {chosen_weights['pred_mlp']:.3f}, "
      f"seq2seq = {chosen_weights['pred_seq']:.3f}")
print(f"CV-WAPE       : {chosen_wape:.4f}")
print(f"Δ vs LGBM solo: {delta_vs_lgbm:+.4f}")

if ships:
    print(f"\n✅ Blend clears the {GAIN_OVER_LGBM_FLOOR} WAPE noise floor — worth shipping as the DL-track submission.")
else:
    print(f"\n✗  Blend gain over LGBM solo is below the {GAIN_OVER_LGBM_FLOOR} noise floor. Do not ship.")
    print("   Re-check the optimizer or the OOF files; the 50/50 baseline in §3 already cleared this bar,")
    print("   so a failed optimal blend would indicate a bug.")

# Save the chosen blend's OOF predictions for the submission notebook.
df_out = df[["series_id", "timestamp", "fold", "lead_h", "target", "pred_lgbm", "pred_mlp", "pred_seq"]].copy()
df_out["pred_blend"] = (
    chosen_weights["pred_lgbm"] * df_out["pred_lgbm"]
    + chosen_weights["pred_mlp"]  * df_out["pred_mlp"]
    + chosen_weights["pred_seq"]  * df_out["pred_seq"]
)
df_out["w_lgbm"] = chosen_weights["pred_lgbm"]
df_out["w_mlp"]  = chosen_weights["pred_mlp"]
df_out["w_seq"]  = chosen_weights["pred_seq"]

out_path = OUTPUT_DIR / "oof_predictions_7b_blend.csv"
df_out.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")
print(f"  rows: {len(df_out):,}, columns: {list(df_out.columns)}")

DECISION
Chosen blend  : 2-way (LGBM + MLP)
Weights       : LGBM = 0.285, MLP = 0.715, seq2seq = 0.000
CV-WAPE       : 14.4683
Δ vs LGBM solo: +0.3136

✅ Blend clears the 0.05 WAPE noise floor — worth shipping as the DL-track submission.

Saved → /kaggle/working/oof_predictions_7b_blend.csv
  rows: 96,768, columns: ['series_id', 'timestamp', 'fold', 'lead_h', 'target', 'pred_lgbm', 'pred_mlp', 'pred_seq', 'pred_blend', 'w_lgbm', 'w_mlp', 'w_seq']


---
## §8. What to do next based on the §7 result

- If §7 chose the **2-way** blend and the gain over LGBM clears 0.05 WAPE: the path forward is a submission notebook that refits LGBM + MLP on the *full* train and writes a validation prediction using the fitted weights. seq2seq becomes a diagnostic-only artifact.
- If §7 chose the **3-way** blend: same submission notebook but includes seq2seq inference as well. Bigger compute cost, justified by the WAPE gain.
- If §6 showed a meaningful gap between horizon-conditional and global weights (> 0.1 WAPE): worth building a proper inner-CV horizon-conditional blend, since the lead-1h regime is the only place seq2seq has a solo edge.
- If §7 didn't ship (gain < 0.05): something is wrong — the naive 50/50 baseline in §3 already beats LGBM by ~0.28. Re-check the optimizer or the input CSVs before doing anything else.